In [98]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

In [99]:
def f_main(x):
    x1, x2 = x
    return -5*x1 - 2*x2 + x1**2 - x1*x2 + x2**2

In [100]:
def penalty_inner(x, r):
    x1, x2 = x

    # Важно: P определена только внутри области g1>0, x1>0, x2>0
    g = 15 - 2*x1 - 3*x2
    h = x1 + 2*x2 - 8

    if g <= 0 or x1 <= 0 or x2 <= 0:
        return np.inf

    return (
        f_main(x)
        + (1/np.sqrt(r)) * (h**2)
        + r * (1/g + 1/x1 + 1/x2)
    )

In [101]:
def grad_penalty(x, r):
    x1, x2 = x
    g = 15 - 2*x1 - 3*x2
    h = x1 + 2*x2 - 8

    dPdx1 = (
        (-5 + 2*x1 - x2)
        + (2/np.sqrt(r)) * h
        + (2*r)/(g**2)
        - r/(x1**2)
    )
    dPdx2 = (
        (-2 - x1 + 2*x2)
        + (4/np.sqrt(r)) * h
        + (3*r)/(g**2)
        - r/(x2**2)
    )

    return np.array([dPdx1, dPdx2], dtype=float)

# _grad_penalty = jax.grad(penalty_inner, argnums=0)

# def grad_penalty(x, r):
#     return np.array(_grad_penalty(jnp.array(x), r))

In [102]:
def gradient_descent_penalty(start, r, alpha0=1.0, eps=1e-6, max_iter=1000):
    x = np.array(start, dtype=float)
    history_x = [x.copy()]
    history_f = [f_main(x)]
    history_P = [penalty_inner(x, r)]
    
    for _ in range(max_iter):
        grad = grad_penalty(x, r)
        grad_norm = np.linalg.norm(grad)
        if grad_norm < eps:
            break
        
        alpha = alpha0
        P_curr = penalty_inner(x, r)
        while True:
            x_new = x - alpha * grad / grad_norm
            P_new = penalty_inner(x_new, r)
            if P_new < P_curr:
                break
            alpha *= 0.5
            if alpha < 1e-12:
                break
        x = x_new
        history_x.append(x.copy())
        history_f.append(f_main(x))
        history_P.append(penalty_inner(x, r))
    
    return x, history_x, history_f, history_P

In [103]:
def interior_point_method(start, r0=10.0, Q=1e-6, E=1e-8, eps_grad=1e-6):
    r = r0
    x = np.array(start, dtype=float)
    global_history_x = []
    global_history_f = []
    global_history_P = []
    global_history_r = []

    while True:
        x_opt, hist_x, hist_f, hist_P = gradient_descent_penalty(x, r, eps=eps_grad)

        for xx, ff, PP in zip(hist_x, hist_f, hist_P):
            global_history_x.append(xx)
            global_history_f.append(ff)
            global_history_P.append(PP)
            global_history_r.append(r)

        P_val = hist_P[-1]
        f_val = hist_f[-1]
        print(f"r = {r:.6f}, x = ({x_opt[0]:.6f}, {x_opt[1]:.6f}), f = {f_val:.6f}, P = {P_val:.6f}")

        if abs(P_val - f_val) <= Q or r < E:
            break
        r /= 4.0
        x = x_opt

    return x_opt, f_val, global_history_x, global_history_f, global_history_P, global_history_r

In [104]:
start_point_1 = (5.0, 1.0)
x_opt_main_1, f_opt_main_1, hist_x_1, hist_f_1, hist_P_1, hist_r_1 = interior_point_method(start_point_1)

r = 10.000000, x = (2.997523, 2.098731), f = -12.086256, P = -0.088728
r = 2.500000, x = (3.229509, 2.220914), f = -12.399648, P = -9.100522
r = 0.625000, x = (3.371129, 2.301091), f = -12.555571, P = -11.636227
r = 0.156250, x = (3.427447, 2.311641), f = -12.592467, P = -12.343663
r = 0.039062, x = (3.435516, 2.301085), f = -12.587402, P = -12.520001
r = 0.009766, x = (3.433510, 2.293494), f = -12.580167, P = -12.561016
r = 0.002441, x = (3.431341, 2.289568), f = -12.575907, P = -12.569983
r = 0.000610, x = (3.430023, 2.287625), f = -12.573686, P = -12.571637
r = 0.000153, x = (3.429312, 2.286665), f = -12.572560, P = -12.571765
r = 0.000038, x = (3.428950, 2.286188), f = -12.571996, P = -12.571654
r = 0.000010, x = (3.428770, 2.285946), f = -12.571712, P = -12.571556
r = 0.000002, x = (3.428682, 2.285825), f = -12.571570, P = -12.571496
r = 0.000001, x = (3.428638, 2.285764), f = -12.571499, P = -12.571463
r = 0.000000, x = (3.428616, 2.285733), f = -12.571464, P = -12.571446
r = 0.0

In [105]:
penalty_inner(start_point_1, 10)

np.float64(11.316227766016837)

In [106]:
f_main(start_point_1)

-6.0

In [107]:
import plotly.graph_objects as go

In [108]:
x1 = [p[0] for p in hist_x_1]
x2 = [p[1] for p in hist_x_1]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=hist_f_1,
    mode='lines+markers',                     # линии + маркеры
    marker=dict(size=5, color='blue', symbol='circle'),
    line=dict(color='blue', width=2),         # стиль линии
    name='f(x)'
))

fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=hist_P_1,
    mode='lines+markers',                     # линии + маркеры
    marker=dict(size=5, color='red', symbol='diamond'),
    line=dict(color='red', width=2),          # стиль линии
    name='P(x, r)'
))

fig.update_layout(
    scene=dict(
        xaxis_title='x₁',
        yaxis_title='x₂',
        zaxis_title='Значение функции'
    ),
    title='Сходимость f и P',
    legend=dict(x=0.8, y=0.9)
)
# Показ в браузере (не требует nbformat)
import plotly.io as pio

pio.renderers.default = "browser"
fig.show()

In [109]:
start_point_2 = (2.0, 4.0)
x_opt_main_2, f_opt_main_2, hist_x_2, hist_f_2, hist_P_2, hist_r_2 = interior_point_method(start_point_2)

r = 10.000000, x = (2.997524, 2.098731), f = -12.086256, P = -0.088728
r = 2.500000, x = (3.229510, 2.220914), f = -12.399648, P = -9.100522
r = 0.625000, x = (3.371128, 2.301091), f = -12.555571, P = -11.636227
r = 0.156250, x = (3.427447, 2.311641), f = -12.592467, P = -12.343663
r = 0.039062, x = (3.435516, 2.301085), f = -12.587402, P = -12.520001
r = 0.009766, x = (3.433510, 2.293494), f = -12.580167, P = -12.561016
r = 0.002441, x = (3.431341, 2.289568), f = -12.575907, P = -12.569983
r = 0.000610, x = (3.430023, 2.287625), f = -12.573686, P = -12.571637
r = 0.000153, x = (3.429313, 2.286665), f = -12.572560, P = -12.571765
r = 0.000038, x = (3.428950, 2.286188), f = -12.571996, P = -12.571654
r = 0.000010, x = (3.428770, 2.285946), f = -12.571712, P = -12.571556
r = 0.000002, x = (3.428682, 2.285825), f = -12.571570, P = -12.571496
r = 0.000001, x = (3.428638, 2.285764), f = -12.571499, P = -12.571463
r = 0.000000, x = (3.428616, 2.285733), f = -12.571464, P = -12.571446
r = 0.0

In [110]:
f_main(start_point_2)

-6.0

In [111]:
penalty_inner(start_point_2, 10)

inf

In [112]:
start_point_3 = (6.0, 4.0)
x_opt_main_3, f_opt_main_3, hist_x_3, hist_f_3, hist_P_3, hist_r_3 = interior_point_method(start_point_3)

r = 10.000000, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 2.500000, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.625000, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.156250, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.039062, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.009766, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.002441, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000610, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000153, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000038, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000010, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000002, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000001, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000000, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000000, x = (6.000000, 4.000000), f = -10.000000, P = inf
r = 0.000000, x = (6.000000, 4.000000),

In [113]:
f_main(start_point_3)

-10.0

In [114]:
penalty_inner(start_point_3, 10)

inf

In [115]:
x1_1 = [p[0] for p in hist_x_1]
x2_1 = [p[1] for p in hist_x_1]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x1_1, y=x2_1, z=hist_f_1,
    mode='lines',                    
    line=dict(color='#FF0000', width=2),        
    name='1 f(x)'
))

fig.add_trace(go.Scatter3d(
    x=x1_1, y=x2_1, z=hist_P_1,
    mode='lines',                    
    line=dict(color='#8B0000', width=2),         
    name='1 P(x, r)'
))

x1_2 = [p[0] for p in hist_x_2]
x2_2 = [p[1] for p in hist_x_2]


fig.add_trace(go.Scatter3d(
    x=x1_2, y=x2_2, z=hist_f_2,
    mode='lines',                   
    line=dict(color='#1E90FF', width=2),       
    name='2 f(x)'
))

fig.add_trace(go.Scatter3d(
    x=x1_2, y=x2_2, z=hist_P_2,
    mode='lines',                 
    line=dict(color='#00008B', width=2),     
    name='2 P(x, r)'
))

x1_3 = [p[0] for p in hist_x_3]
x2_3 = [p[1] for p in hist_x_3]

fig.add_trace(go.Scatter3d(
    x=x1_3, y=x2_3, z=hist_f_3,
    mode='lines',                  
    line=dict(color='#228B22', width=2), 
    name='3 f(x)'
))

fig.add_trace(go.Scatter3d(
    x=x1_3, y=x2_3, z=hist_P_3,
    mode='lines',                  
    line=dict(color='#006400', width=2),   
    name='3 P(x, r)'
))
fig.update_layout(
    scene=dict(
        xaxis_title='x₁',
        yaxis_title='x₂',
        zaxis_title='Значение функции'
    ),
    title='Сходимость f и P',
    legend=dict(x=0.8, y=0.9)
)

all_x1 = x1_1 + x1_2 + x1_3  
all_x2 = x2_1 + x2_2 + x2_3
x_min, x_max = min(all_x1), max(all_x1)
y_min, y_max = min(all_x2), max(all_x2)

margin_x = 0.1 * (x_max - x_min) if x_max != x_min else 1
margin_y = 0.1 * (y_max - y_min) if y_max != y_min else 1

# 2. Создаём сетку для поверхности f(x1, x2)
x_grid = np.linspace(x_min - margin_x, x_max + margin_x, 80)
y_grid = np.linspace(y_min - margin_y, y_max + margin_y, 80)
X, Y = np.meshgrid(x_grid, y_grid)

# f(x1, x2) = -5*x1 - 2*x2 + x1^2 - x1*x2 + x2^2
Z = -5*X - 2*Y + X**2 - X*Y + Y**2

# Для go.Surface удобнее передавать 1D x/y и 2D z
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=Z,
    colorscale='Viridis',
    opacity=0.55,
    showscale=True,
    name='f(x) (поверхность)'
))

fig.show()